In [1]:
import numpy as np
import pandas as pd
import os
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectPercentile
from sklearn.model_selection import RepeatedStratifiedKFold, cross_validate
from sklearn.feature_selection import f_classif
from sklearn.pipeline import Pipeline
import joblib

In [ ]:
from pathlib import Path

base_dir = Path.cwd()
OUTPUT_DIR = base_dir / "output"
if not OUTPUT_DIR.exists():
    OUTPUT_DIR.mkdir()

DATASETS_PATH = "/home/mariya/Data/pallas/mlomix_datasets"
TRAIN_DATASET_PATH = os.path.join(DATASETS_PATH, "train-dnam")
X_train_path = os.path.join(TRAIN_DATASET_PATH, "features.dnam.csv")
y_train_path = os.path.join(TRAIN_DATASET_PATH, "labels.csv")

RANDOM_SEED = 42
N_JOBS = 4

FEATURE_SELECTION_PERCENTILE = 10  # KEEP TOP n% OF CPG SITES

In [3]:
# Read in X_train and y_train
X_train = pd.read_csv(X_train_path, index_col=0)
X_train = X_train.astype(np.float32)

y_train = pd.read_csv(y_train_path, index_col=0)

X_train, y_train

(              cg00000029  cg00000109  cg00000165  cg00000236  cg00000289  \
 id                                                                         
 ALL_1277          0.7557      0.9169      0.0990      0.9222      0.7708   
 ALL_1286          0.8646      0.8970      0.1051      0.9038      0.6404   
 ALL_1300          0.8852      0.8994      0.0783      0.8878      0.6192   
 ALL_1322          0.7903      0.9051      0.2426      0.9112      0.6721   
 ALL_1352          0.9116      0.8630      0.1516      0.8653      0.7713   
 ...                  ...         ...         ...         ...         ...   
 TCGA-AB-2991      0.3880      0.9099      0.1818      0.5858      0.8566   
 TCGA-AB-2992      0.9088      0.9207      0.1163      0.8856      0.7777   
 TCGA-AB-2997      0.2163      0.9154      0.1116      0.4988      0.8602   
 TCGA-AB-3007      0.7086      0.8974      0.1788      0.5349      0.8395   
 TCGA-AB-3012      0.5843      0.9399      0.1827      0.6158      0.8592   

In [4]:
# Show class distributions in y_train and y_test
print("y_train class distribution:")
print(y_train.value_counts())

y_train class distribution:
class                   
AML, KMT2A-r                241
B-ALL, high hyperdiploid    229
B-ALL, ETV6::RUNX1          199
AML, RUNX1::RUNX1T1         174
AML, NPM1alt                130
AML, CBFB::MYH11            127
Non-leukemic                 95
T-ALL                        73
AML, NUP98-r                 69
AML, CEBPAalt                55
B-ALL, PAX5alt               47
B-ALL, KMT2A-r               43
B-ALL, TCF3::PBX1            26
B-ALL, DUX4-r                22
AML, CBFA2T3::GLIS2          22
AML, DEK::NUP214             19
B-ALL, BCR::ABL1             17
AML, APL-PML::RARA           16
AML, ETV6-r                  16
B-ALL, ETV6::RUNX1-like      15
B-ALL, iAMP21                15
B-ALL, ZNF384-r              15
B-ALL, BCR::ABL1-like        14
AML, KAT6A-r                  8
AML, PICALM::MLLT10           6
AML, FUS::ERG                 6
B-ALL, MEF2D-r                6
B-ALL, PAX5 P80R              6
Name: count, dtype: int64


In [5]:
y_train = y_train.iloc[:, 0].values  # Convert to 1D numpy array
y_train

array(['B-ALL, high hyperdiploid', 'B-ALL, high hyperdiploid',
       'B-ALL, ETV6::RUNX1', ..., 'AML, APL-PML::RARA',
       'AML, APL-PML::RARA', 'AML, APL-PML::RARA'],
      shape=(1711,), dtype=object)

In [6]:
n_splits = 5

cv = RepeatedStratifiedKFold(
    n_splits=n_splits,
    n_repeats=1,
    random_state=RANDOM_SEED
)

rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=40,
    max_features=0.25,
    min_samples_leaf=1,
    class_weight="balanced",
    random_state=RANDOM_SEED,
    n_jobs=1,  # important when parallelizing CV
)

cv_results = cross_validate(
    rf,
    X_train,
    y_train,
    cv=cv,
    scoring="f1_macro",
    n_jobs=N_JOBS,
    verbose=2,
    return_train_score=True,
    return_estimator=True
)

train_mean = cv_results['train_score'].mean()
train_std  = cv_results['train_score'].std()
val_mean   = cv_results['test_score'].mean()
val_std    = cv_results['test_score'].std()

print(f"Train F1-macro: {train_mean:.3f} ± {train_std:.3f}")
print(f"Val   F1-macro: {val_mean:.3f} ± {val_std:.3f}")
print(f"Gap:            {train_mean - val_mean:.3f}")

[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.


[CV] END ................................................... total time=128.4min
[CV] END ................................................... total time=129.9min
[CV] END ................................................... total time=136.2min
[CV] END ................................................... total time=136.7min
[CV] END ................................................... total time=101.8min
Train F1-macro: 1.000 ± 0.000
Val   F1-macro: 0.808 ± 0.032
Gap:            0.192


[Parallel(n_jobs=4)]: Done   5 out of   5 | elapsed: 230.3min finished


In [ ]:
feature_names = X_train.columns

importances = []

for est in cv_results["estimator"]:
    importances.append(est.feature_importances_)

importances = np.vstack(importances)

importance_df = pd.DataFrame(
    importances,
    columns=feature_names
)
importance_df

,cg00000029,cg00000109,cg00000165,cg00000236,cg00000289,cg00000292,cg00000321,cg00000363,cg00000714,cg00001099,...,ch.4.1680698F,ch.4.2662600F,ch.4.3427878R,ch.6.115952F,ch.6.1693624F,ch.6.92327598R,ch.7.130270068F,ch.7.1839793F,ch.9.2043201R,ch.9.31239617R
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.000000,0.0,0.000000,0.000007,0.000000,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.000000,0.0,0.000028,0.000000,0.000003,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000022,...,0.0,0.0,0.0,0.000008,0.0,0.000000,0.000000,0.000000,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.0


In [13]:
output_path = os.path.join(OUTPUT_DIR, "preliminary_feature_importances.csv")
importance_df.to_csv(output_path)

In [17]:
top_k = int(X_train.shape[1] * 0.10)

selected_per_fold = []

for row in importances:

    idx = np.argsort(row)[::-1][:top_k]

    selected_per_fold.append(
        feature_names[idx]
    )

from collections import Counter

counter = Counter()

for features in selected_per_fold:
    counter.update(features)

freq_df = pd.DataFrame(
    counter.items(),
    columns=["feature", "count"]
)

freq_df = freq_df.sort_values(
    "count",
    ascending=False
)
freq_path=os.path.join(OUTPUT_DIR, "preliminary_feature_frequencies.csv")
freq_df.to_csv(freq_path, index=False)

In [32]:
# Save all features with freq >= 2
selected_features = freq_df[freq_df["count"] >= 1]["feature"].tolist()
selected_features_path = os.path.join(OUTPUT_DIR, "top_features.txt")

with open(selected_features_path, 'w') as f:
    for feature in selected_features:
        f.write(f"{feature}\n")

print(f"Selected {len(selected_features)} features with frequency >= 2. Saved to {selected_features_path}.")

Selected 49632 features with frequency >= 2. Saved to /home/mariya/Development/pallas/training/dnam/output/top_features.txt.


In [33]:
# Sanity check, train a model on the reduced feature space
X_reduced = X_train[selected_features]
X_reduced

,cg06745684,cg16034168,cg08166720,cg26992245,cg11161853,cg08616111,cg00646731,cg11367658,cg00015639,cg21359950,...,cg25533943,cg02265239,cg01868791,cg19364777,cg06531475,cg06532037,cg06531158,cg06534208,cg06533798,cg06533788
id,,,,,,,,,,,,,,,,,,,,,
ALL_1277,0.6023,0.9639,0.9205,0.8448,0.0287,0.9182,0.9248,0.9490,0.8169,0.7564,...,0.0269,0.7944,0.9062,0.3360,0.3525,0.0117,0.8655,0.9531,0.6822,0.9696
ALL_1286,0.7584,0.9572,0.9278,0.8573,0.0294,0.9339,0.9331,0.9371,0.9223,0.5278,...,0.0347,0.7144,0.9206,0.3976,0.5422,0.0305,0.8066,0.9571,0.7134,0.9612
ALL_1300,0.8942,0.9654,0.9193,0.8789,0.0269,0.9211,0.9334,0.9435,0.8866,0.6776,...,0.0412,0.7074,0.9461,0.3644,0.3741,0.2925,0.7968,0.9655,0.6501,0.9718
ALL_1322,0.8010,0.9525,0.9410,0.6207,0.0257,0.8110,0.9162,0.9391,0.8046,0.7715,...,0.0313,0.7727,0.8101,0.3577,0.2680,0.4037,0.8326,0.9407,0.4010,0.9692
ALL_1352,0.3665,0.9360,0.9238,0.8766,0.6753,0.8626,0.9031,0.9211,0.7714,0.5609,...,0.0536,0.7702,0.7304,0.3120,0.4785,0.0562,0.6898,0.9045,0.1854,0.9626
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TCGA-AB-2991,0.8754,0.9817,0.9702,0.9372,0.0225,0.9488,0.9165,0.9501,0.8893,0.9186,...,0.0245,0.9175,0.8833,0.2663,0.7949,0.0146,0.9374,0.9515,0.9003,0.9820
TCGA-AB-2992,0.8416,0.9771,0.9639,0.9356,0.0376,0.9279,0.8234,0.9150,0.8539,0.9020,...,0.4530,0.8127,0.9345,0.2448,0.2425,0.0149,0.9057,0.9171,0.8887,0.9717
TCGA-AB-2997,0.5491,0.9818,0.9713,0.9561,0.0288,0.9639,0.9355,0.9467,0.9287,0.9009,...,0.0206,0.9359,0.5433,0.1043,0.9128,0.0117,0.9373,0.9592,0.8676,0.9857


In [34]:
n_splits = 5

cv = RepeatedStratifiedKFold(
    n_splits=n_splits,
    n_repeats=1,
    random_state=RANDOM_SEED
)

rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=40,
    max_features=0.25,
    min_samples_leaf=1,
    class_weight="balanced",
    random_state=RANDOM_SEED,
    n_jobs=1,  # important when parallelizing CV
)

cv_results = cross_validate(
    rf,
    X_reduced,
    y_train,
    cv=cv,
    scoring="f1_macro",
    n_jobs=N_JOBS,
    verbose=2,
    return_train_score=True,
    return_estimator=True
)

train_mean = cv_results['train_score'].mean()
train_std  = cv_results['train_score'].std()
val_mean   = cv_results['test_score'].mean()
val_std    = cv_results['test_score'].std()

print(f"Train F1-macro: {train_mean:.3f} ± {train_std:.3f}")
print(f"Val   F1-macro: {val_mean:.3f} ± {val_std:.3f}")
print(f"Gap:            {train_mean - val_mean:.3f}")

[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.


[CV] END .................................................... total time=32.3min
[CV] END .................................................... total time=34.6min
[CV] END .................................................... total time=35.0min
[CV] END .................................................... total time=35.1min
[CV] END .................................................... total time=23.2min
Train F1-macro: 1.000 ± 0.000
Val   F1-macro: 0.817 ± 0.029
Gap:            0.183


[Parallel(n_jobs=4)]: Done   5 out of   5 | elapsed: 55.5min finished


In [35]:
feature_names = X_reduced.columns

importances = []

for est in cv_results["estimator"]:
    importances.append(est.feature_importances_)

importances = np.vstack(importances)

importance_df = pd.DataFrame(
    importances,
    columns=feature_names
)
importance_df

,cg06745684,cg16034168,cg08166720,cg26992245,cg11161853,cg08616111,cg00646731,cg11367658,cg00015639,cg21359950,...,cg25533943,cg02265239,cg01868791,cg19364777,cg06531475,cg06532037,cg06531158,cg06534208,cg06533798,cg06533788
0,0.000533,0.004681,0.004752,0.003103,0.001975,0.002166,0.002863,0.003156,0.006423,0.003380,...,0.000000,0.0,0.0,0.000007,0.0,0.0,0.0,0.00000,0.000000,0.000000e+00
1,0.000000,0.001482,0.004526,0.002197,0.001328,0.004290,0.002021,0.000650,0.004219,0.003050,...,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.00002,0.000006,0.000000e+00
2,0.000038,0.002088,0.004958,0.001176,0.001477,0.003772,0.006902,0.001989,0.007737,0.001710,...,0.000000,0.0,0.0,0.000054,0.0,0.0,0.0,0.00000,0.000000,0.000000e+00
3,0.000482,0.000563,0.004664,0.002518,0.000000,0.008636,0.003411,0.000046,0.006587,0.000354,...,0.000000,0.0,0.0,0.000470,0.0,0.0,0.0,0.00000,0.000000,1.471067e-19
4,0.000221,0.000249,0.002918,0.002459,0.003509,0.002703,0.001440,0.002545,0.004819,0.003219,...,0.000004,0.0,0.0,0.000000,0.0,0.0,0.0,0.00000,0.000000,0.000000e+00


In [36]:
output_path = os.path.join(OUTPUT_DIR, "reduced_feature_importances.csv")
importance_df.to_csv(output_path)

In [37]:
top_k = int(X_train.shape[1] * 0.10)

selected_per_fold = []

for row in importances:

    idx = np.argsort(row)[::-1][:top_k]

    selected_per_fold.append(
        feature_names[idx]
    )

from collections import Counter

counter = Counter()

for features in selected_per_fold:
    counter.update(features)

freq_df = pd.DataFrame(
    counter.items(),
    columns=["feature", "count"]
)

freq_df = freq_df.sort_values(
    "count",
    ascending=False
)
freq_path=os.path.join(OUTPUT_DIR, "reduced_feature_frequencies.csv")
freq_df.to_csv(freq_path, index=False)

In [44]:
# Save all features with freq >= 2
selected_features_df = freq_df[freq_df["count"] >= 1]
selected_features_df

,feature,count
1,cg07232063,5
2,cg15172966,5
9538,cg22711792,5
9539,cg16845486,5
9540,cg06714655,5
...,...,...
36805,cg07948095,1
36804,cg07912416,1
36831,cg07948034,1
36830,cg07909178,1


In [46]:
selected_features = selected_features_df["feature"].tolist()
selected_features_path = os.path.join(OUTPUT_DIR, "top_features_reduced.txt")

with open(selected_features_path, 'w') as f:
    for feature in selected_features:
        f.write(f"{feature}\n")

print(f"Selected {len(selected_features)} features with frequency >= 1. Saved to {selected_features_path}.")

Selected 36845 features with frequency >= 1. Saved to /home/mariya/Development/pallas/training/dnam/output/top_features_reduced.txt.
